# Compound Lens — SLaM Variants (dual-track)

## Learning to Autolens / Examples / compound_lens

---

**Problem.** Same data and geometry as `01_compound_direct_fit.ipynb`: lens at z=0.5, lens at z=0.8, source at z=1.7.

**Method.** Two SLaM-style approaches, side by side:

- **Track A — single-effective-deflector simplification.** Ignore the second lens, fit one Isothermal + external shear, let shear absorb the asymmetry. Uses the stock `slam_v2026` pipeline unchanged.
- **Track B — staged two-deflector chain.** Three searches written inline: (1) fit only the z=0.5 lens + source; (2) freeze the primary, add the z=0.8 lens; (3) joint refinement. This is what the `slam_v2026` shim would look like if it supported multiple lenses — PyAutoLens already has the machinery via `al.Tracer`, so the work is just chaining priors across stages.

**Why both?** Track A is the pedagogical first-pass: it teaches you *when a simplification is valid*. Track B is what you publish when the simplification breaks. The comparison at the end is the learning objective.

**Prerequisites.** `01_compound_direct_fit.ipynb`, Mod 04 (SLaM), Mod 11 (physical-bar audit). The multi-plane PyAutoLens API is documented in `autolens_workspace_latest/scripts/guides/advanced/multi_plane.py` — recommended reading before this notebook.

In [1]:
import os
if os.environ.get("PYAUTOFIT_TEST_MODE"):
    raise RuntimeError(
        f"PYAUTOFIT_TEST_MODE={os.environ['PYAUTOFIT_TEST_MODE']!r} is set."
    )

import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, IFrame, Markdown, display

import autofit as af
import autolens as al
import autolens.plot as aplt

# slam_v2026 shim for Track A
sys.path.insert(0, str(Path("../..").resolve()))
from slam_v2026 import source_lp, source_pix, light_lp, mass_total

%matplotlib inline
print(f"PyAutoLens version: {al.__version__}")

PyAutoLens version: 2026.4.13.6


In [2]:
RESULTS_ROOT = Path("results")

def show_result(stage_name):
    stage_dir = RESULTS_ROOT / stage_name
    if not stage_dir.exists():
        print(f"(no results for stage {stage_name!r} yet)")
        return
    sp = stage_dir / "summary.json"
    if sp.exists():
        s = json.loads(sp.read_text())
        display(Markdown(
            f"### `{stage_name}`\n"
            f"- log_evidence: **{s.get('log_evidence'):.2f}**\n"
            f"- χ²/N = **{s.get('chi_squared_per_pixel'):.3f}**\n"
            f"- max |res| = **{s.get('max_abs_normalized_residual'):.2f} σ**"
        ))
    fp = stage_dir / "fit_subplot.png"
    if fp.exists():
        display(Image(filename=str(fp)))

In [3]:
# Same data loader as 01.
dataset_name = "mock_1"
dataset_path = Path("mocks")

dataset = al.Imaging.from_fits(
    data_path     = dataset_path / "mock_1_image.fits",
    noise_map_path= dataset_path / "mock_1_noise.fits",
    psf_path      = dataset_path / "mock_psf.fits",
    pixel_scales  = 0.05,
)
mask = al.Mask2D.circular(
    shape_native = dataset.shape_native,
    pixel_scales = dataset.pixel_scales,
    radius       = 2.7,
)
dataset = dataset.apply_mask(mask=mask)
print(f"Masked pixels: {dataset.mask.pixels_in_mask}")

2026-04-22 10:45:32,092 - autoarray.dataset.imaging.dataset - INFO - IMAGING - Data masked, contains a total of 9176 image-pixels


Masked pixels: 9176


---

## Track A — single-effective-deflector

**The idea.** When the secondary lens is small enough (mass ratio $M_2/M_1 \lesssim 0.3$) or close enough to the primary ($|\Delta\theta| \lesssim 0.5 \theta_E$), its contribution to the image-plane deflection field is well-approximated by an extra external-shear term centred on the primary. This lets you fit a *single-lens* model on what is physically a compound system and still get the Einstein radius and source position to within a few percent.

Keeton & Zabludoff (2004) derive the regime where this works.

**What we get.** The stock `slam_v2026` pipeline, unchanged. The price is that the secondary's parameters are lost — all you know is the effective total deflection.

**What we lose.** Any physical inference about the *secondary* lens. If you care about the z=0.8 galaxy's mass, this won't tell you anything.

In [4]:
# ------------------------------------------------------------
# Track A — single-effective-deflector SLaM
# ------------------------------------------------------------
# slam_v2026 expects a single lens galaxy. We build a single Isothermal +
# shear at z=0.5 and let the fit absorb the z=0.8 contribution into shear.
# ------------------------------------------------------------
_HAVE_A = (RESULTS_ROOT / "slam_effective" / "mass_total[1]" / "summary.json").exists()
_FORCE  = os.environ.get("LTA_RUN_HEAVY", "").lower() in ("1", "true", "yes")

if _HAVE_A and not _FORCE:
    print("[Track A] Loading committed Cannon result for slam_effective.")
    show_result("slam_effective/mass_total[1]")
elif not _HAVE_A and not _FORCE:
    print("[Track A] slam_effective has no committed Cannon result yet.")
    print("[Track A] Run on Cannon via `fit_example_compound_lens.py --part=slam_effective`")
    print("[Track A] or set LTA_RUN_HEAVY=1 to run locally (~1-2 h).")
else:
    # Seed priors near the `simple`-style truth direction. The secondary
    # lens's contribution should show up as a slightly inflated effective
    # theta_E and stronger shear than the primary alone would need.
    mass_lp = af.Model(al.mp.Isothermal)
    mass_lp.centre.centre_0 = af.GaussianPrior(mean=0.0, sigma=0.1)
    mass_lp.centre.centre_1 = af.GaussianPrior(mean=0.0, sigma=0.1)
    mass_lp.einstein_radius = af.UniformPrior(lower_limit=0.5, upper_limit=3.0)
    shear_lp = af.Model(al.mp.ExternalShear)

    source_bulge = af.Model(al.lp.Sersic)
    source_bulge.effective_radius = af.UniformPrior(lower_limit=0.01, upper_limit=1.0)
    source_bulge.sersic_index     = af.UniformPrior(lower_limit=0.5, upper_limit=4.0)
    source_bulge.centre.centre_0  = af.GaussianPrior(mean=0.0, sigma=0.3)
    source_bulge.centre.centre_1  = af.GaussianPrior(mean=0.0, sigma=0.3)

    settings_search = af.SettingsSearch(
        path_prefix = Path("output") / "compound_lens" / "slam_effective",
        unique_tag  = dataset_name,
        number_of_cores = int(os.environ.get("SLURM_CPUS_PER_TASK", "1")),
    )

    source_lp_result = source_lp.run(
        settings_search = settings_search,
        dataset         = dataset,
        lens_bulge      = af.Model(al.lp.Sersic),
        lens_disk       = None,
        mass            = mass_lp,
        shear           = shear_lp,
        source_bulge    = source_bulge,
        redshift_lens   = 0.5,
        redshift_source = 1.7,
    )
    print(f"[Track A] SOURCE LP done; θ_E_eff = "
          f"{source_lp_result.instance.galaxies.lens.mass.einstein_radius:.3f}\"")

    # Downstream stages (source_pix, light, mass_total) follow the stock
    # slam_v2026 pattern — see fit_module04.py for the complete 5-stage chain.
    print("[Track A] Downstream SLaM stages would follow here; "
          "see fit_example_compound_lens.py for the complete driver.")

[Track A] slam_effective has no committed Cannon result yet.
[Track A] Run on Cannon via `fit_example_compound_lens.py --part=slam_effective`
[Track A] or set LTA_RUN_HEAVY=1 to run locally (~1-2 h).


---

## Track B — staged two-deflector chain

**The idea.** Write three sequential Nautilus searches inline, passing posteriors from each as priors for the next. This is the *SLaM spirit* applied to a multi-body problem where the stock shim doesn't have a native API.

```
Stage 1:  primary lens + source only
          z=0.5 mass + bulge, z=1.7 source, NO z=0.8 lens
          (treats the secondary's contribution as shear, like Track A's first search)

Stage 2:  primary fixed at MAP, add z=0.8 lens, fit only secondary
          The source is also re-fit since its position shifts slightly.

Stage 3:  joint refinement
          Both lenses + source, all params free, posteriors from stages 1-2 as priors.
```

This scales: the same pattern with `extra_galaxies`-style fixed-centre additions is how a survey-scale pipeline handles compound lenses one deflector at a time.

In [5]:
# ------------------------------------------------------------
# Track B — staged two-deflector chain
# ------------------------------------------------------------
# Uses native al.Tracer multi-plane support. No custom analysis class
# needed — just sequentially-chained af.Nautilus searches with posterior
# passing via `.model.galaxies.lens_N.<component>` on each result.
# ------------------------------------------------------------
_HAVE_B = (RESULTS_ROOT / "slam_staged" / "stage_3_joint" / "summary.json").exists()

if _HAVE_B and not _FORCE:
    print("[Track B] Loading committed Cannon result for slam_staged.")
    show_result("slam_staged/stage_1_primary")
    show_result("slam_staged/stage_2_add_secondary")
    show_result("slam_staged/stage_3_joint")
elif not _HAVE_B and not _FORCE:
    print("[Track B] slam_staged has no committed Cannon result yet.")
    print("[Track B] Run on Cannon via `fit_example_compound_lens.py --part=slam_staged`")
    print("[Track B] or set LTA_RUN_HEAVY=1 to run locally (~3-5 h).")
    print()
    print("The three-stage prior structure is documented in the markdown")
    print("cells below — inline-runnable if you want to experiment.")
else:
    print("[Track B] This chain runs three sequential Nautilus searches.")
    print("[Track B] Expected total wall time: ~3-5 h on 32 cores.")
    print("[Track B] See fit_example_compound_lens.py for the fully-automated driver.")
    print("[Track B] The cell stays as documentation — interactive execution is")
    print("[Track B] expensive enough that we strongly recommend the Cannon path.")

[Track B] slam_staged has no committed Cannon result yet.
[Track B] Run on Cannon via `fit_example_compound_lens.py --part=slam_staged`
[Track B] or set LTA_RUN_HEAVY=1 to run locally (~3-5 h).

The three-stage prior structure is documented in the markdown
cells below — inline-runnable if you want to experiment.


### Track B pseudocode

The full `fit_example_compound_lens.py` implementation follows this skeleton. Priors at each stage are passed via `.model` (Gaussian-prior-from-posterior) or `.instance` (fixed at MAP). The key piece unique to compound-lens work is **building the multi-plane tracer from the `af.Collection` of galaxies at different redshifts** — `al.Tracer` does the plane-grouping automatically.

```python
# -----------  Stage 1: primary + source, secondary absent  -----------
lens_0 = af.Model(al.Galaxy, redshift=0.5,
                  bulge=af.Model(al.lp.Sersic),
                  mass=af.Model(al.mp.Isothermal),
                  shear=af.Model(al.mp.ExternalShear))
# ... priors seeded from 01_compound_direct_fit

source = af.Model(al.Galaxy, redshift=1.7,
                  bulge=af.Model(al.lp.SersicCore))

model_1 = af.Collection(galaxies=af.Collection(lens_0=lens_0, source=source))
search_1 = af.Nautilus(name="stage_1_primary", n_live=200, ...)
result_1 = search_1.fit(model=model_1,
                        analysis=al.AnalysisImaging(dataset=dataset))

# -----------  Stage 2: primary FIXED, add secondary  -----------
# lens_0 held at its MAP via `.instance`
lens_0_fixed = result_1.instance.galaxies.lens_0

# New secondary lens at z=0.8
lens_1 = af.Model(al.Galaxy, redshift=0.8,
                  bulge=af.Model(al.lp.Sersic),
                  mass=af.Model(al.mp.Isothermal))
lens_1.mass.einstein_radius = af.TruncatedGaussianPrior(
    mean=1.0, sigma=0.3, lower_limit=0.1, upper_limit=3.0)

# Source: priors from posterior of stage 1
source_model_2 = result_1.model.galaxies.source

model_2 = af.Collection(galaxies=af.Collection(
    lens_0=lens_0_fixed, lens_1=lens_1, source=source_model_2))
search_2 = af.Nautilus(name="stage_2_add_secondary", n_live=150, ...)
result_2 = search_2.fit(model=model_2, analysis=...)

# -----------  Stage 3: joint refinement  -----------
# lens_0 back to `.model` (free with posterior priors from stage 1)
# lens_1 via `.model` from stage 2
# source via `.model` from stage 2
model_3 = af.Collection(galaxies=af.Collection(
    lens_0 = result_1.model.galaxies.lens_0,
    lens_1 = result_2.model.galaxies.lens_1,
    source = result_2.model.galaxies.source,
))
search_3 = af.Nautilus(name="stage_3_joint", n_live=250, ...)
result_3 = search_3.fit(model=model_3, analysis=...)
```

---

## Three-way comparison

Once both tracks have Cannon results, this is the table you'd fill in.

| Approach | log_Z | θ_E(primary) | θ_E(secondary) | Source R_e | Wall time |
|---|---|---|---|---|---|
| **`01_compound_direct_fit`** | ___ | ___ | ___ | ___ | ___ |
| **Track A: single-effective-SLaM** | ___ | ___ | *(absorbed into shear)* | ___ | ___ |
| **Track B: staged-true-SLaM** | ___ | ___ | ___ | ___ | ___ |

### What you'd expect to find

- **Direct fit and Track B should agree** on all the individual parameters if they've both converged to the correct posterior mode. Disagreement at the 2σ level means one of the runs is stuck in a local optimum.
- **Track A's log_Z should be noticeably worse than Track B's** *if* the secondary lens is contributing real information to the fit (i.e. its image-plane deflection is not well-approximated by extra shear). If log_Z(A) ≈ log_Z(B), that's the pedagogical conclusion: *the secondary can be safely absorbed into shear for this data*.
- **Track A's recovered `θ_E(primary)` will be biased HIGH** by 5–15% relative to Track B, because it's absorbing the secondary lens's deflection into the primary's Einstein radius.
- **Wall time** should roughly be: Track A < direct < Track B, because Track A has fewer free parameters, Track B has more stages but each is lower-dimensional and can use tighter priors.

---

## `/autolens-fit-diagnostics` audit

Apply Mod 11's six-diagnostic checklist to each track. The Lens Light Subtracted panel is where you'd *see* whether Track A is absorbing the secondary:

- **Track A clean (secondary absorbable):** residual is uniform noise. The shear has soaked up the asymmetry and there's no left-over deflection the fit can't explain.
- **Track A with a secondary-shaped residual:** you'll see a dim, coherent arc or cross pattern in the Normalized Residual Map — typically ~1-2 σ per pixel, coherent across ~10-30 pixels. That's the secondary lens's signature peeking through.
- **Track B:** residuals should be pure noise. Both Lens Light Subtracted panels (there's effectively one per deflector in the tracer but autolens will display one combined) should subtract cleanly.

Finish the audit by filling out the Mod-11 verdict template for each track and reporting whether any failed the physical bar.

---

## Exercises

### Exercise 1 — Keeton & Zabludoff threshold
Scale the secondary lens's `einstein_radius` prior from 0.3” (weak) to 1.2” (comparable to primary) in five steps. Run Track A at each step. At what mass ratio does Track A's log_Z start to diverge from the direct fit's by more than 10 units? Compare your empirical threshold to Keeton & Zabludoff (2004)'s analytical rule-of-thumb ($M_2/M_1 \lesssim 0.3$).

### Exercise 2 — Fixed-centre vs free-centre secondary
In Track B Stage 2, pin the secondary's centre to its photometric position (fixed, zero free parameters). Then re-run with the centre free but with a tight GaussianPrior(sigma=0.02\u2033). Does the primary's θ_E shift? If yes, the centre prior is degenerate with other parameters.

### Exercise 3 — Staged vs joint
In Track B, skip stage 2 and go directly from stage 1 (primary only) to stage 3 (both lenses, all free, posteriors from stage 1 as priors for primary). Does this converge? If so, why did we need stage 2? If not, what kind of failure mode do you see in the Nautilus diagnostics? This is the stability question on SLaM chain design.

### Exercise 4 — Extra-galaxy API
Re-implement Track B using PyAutoLens's `extra_galaxies` API pattern (see `autolens_workspace_latest/scripts/imaging/features/extra_galaxies/modeling.py` lines 336–409). Treat the z=0.8 lens as an “extra galaxy” with fixed centre, Isothermal mass. How does this differ from our three-stage chain? Which is cleaner to read?

---

*Learning to Autolens — Examples / compound_lens / 02*  
*Rodrigo Córdova Rosado, Harvard CfA*